### Loading the Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
train.head()

In [ ]:
print(train.isnull().sum())

### Data Cleaning And Processing

##### Check for duplicate rows

In [ ]:
print("Duplicate rows in train:", train.duplicated().sum())
print("Duplicate rows in test:", test.duplicated().sum())

##### Filling missing values with average

In [ ]:
mean_delay = train['Arrival Delay in Minutes'].mean()
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(mean_delay)
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(mean_delay)

##### drop columns — removes Unnamed: 0 (a leftover index from the CSV) and id (just a passenger ID, no predictive value)

In [ ]:
train.drop(columns=['Unnamed: 0', 'id'], inplace=True)
test.drop(columns=['Unnamed: 0', 'id'], inplace=True)


##### turning text labels into 0 or 1 

In [ ]:
train['satisfaction'] = train['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})
test['satisfaction'] = test['satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

##### Encode categorical columns to numeric values

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
label_encoders = {}

for col in cat_cols:
    label_encoders[col] = LabelEncoder()
    train[col] = label_encoders[col].fit_transform(train[col])
    test[col] = label_encoders[col].transform(test[col])

# Verify
print("Missing values left:", train.isnull().sum().sum())
print("Shape:", train.shape)
print(train.dtypes)

In [ ]:
train.head(10)

### Feature Engineering

##### 1. Total Delay = Departure Delay + Arrival Delay combined into one feature¶

In [ ]:
train['Total Delay'] = train['Departure Delay in Minutes'] + train['Arrival Delay in Minutes']
test['Total Delay'] = test['Departure Delay in Minutes'] + test['Arrival Delay in Minutes']


##### Average Service Rating across all 14 service columns

In [ ]:
service_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

train['Avg Service Rating'] = train[service_cols].mean(axis=1)
test['Avg Service Rating'] = test[service_cols].mean(axis=1)

##### Verify the new columns exist

In [ ]:
print("New shape:", train.shape)
print(train[['Total Delay', 'Avg Service Rating']].head(10))

In [ ]:
train.head(10)

##### Basic Statistics

In [ ]:
train.describe()

In [ ]:
train.info()

### Exploratory Data Analysis (EDA)

##### Satisfaction distribution

In [ ]:
sns.countplot(x='satisfaction', data=train)
plt.title('Satisfaction Distribution')
plt.xticks([0, 1], ['Dissatisfied', 'Satisfied'])
plt.show()

##### Satisfaction by Travel Class

In [ ]:
sns.countplot(x='Class', hue='satisfaction', data=train)
plt.title('Satisfaction by Travel Class')
plt.xticks([0, 1, 2], ['Business', 'Eco', 'Eco Plus'])
plt.legend(['Dissatisfied', 'Satisfied'])
plt.show()

##### Correlation heatmap of service ratings

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(train[service_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Service Rating Correlations')
plt.tight_layout()
plt.show()

##### Total Delay vs Satisfaction

In [ ]:
sns.boxplot(x='satisfaction', y='Total Delay', data=train)
plt.ylim(0, 200)
plt.xticks([0, 1], ['Dissatisfied', 'Satisfied'])
plt.title('Total Delay vs Satisfaction')
plt.show()

#### Prepare Features for Modeling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) from target (y)
X_train = train.drop(columns=['satisfaction'])
y_train = train['satisfaction']
X_test = test.drop(columns=['satisfaction'])
y_test = test['satisfaction']

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Scale all features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# Verify scaling worked - means should be ~0, std should be ~1
print(X_train_scaled.describe().loc[['mean', 'std']])

### Classification Models

#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:,1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

#### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)

print("=== Decision Tree ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, dt.predict_proba(X_test_scaled)[:,1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, rf.predict_proba(X_test_scaled)[:,1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

### HyperParameter Tuning

#### Tune Logistic Regression

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10]
}

grid_lr = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), 
                       param_grid_lr, cv=3, scoring='accuracy')
grid_lr.fit(X_train_scaled, y_train)

print("=== Logistic Regression Tuning ===")
print("Best C:", grid_lr.best_params_)
print("Best CV Accuracy:", round(grid_lr.best_score_, 4))

#### Tune Decision Tree

In [ ]:
param_grid_dt = {
    'max_depth': [5, 10, 15, 20, None]
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), 
                       param_grid_dt, cv=3, scoring='accuracy')
grid_dt.fit(X_train_scaled, y_train)

print("=== Decision Tree Tuning ===")
print("Best max_depth:", grid_dt.best_params_)
print("Best CV Accuracy:", round(grid_dt.best_score_, 4))

#### Tune Random Forest

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), 
                       param_grid_rf, cv=3, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train)

print("=== Random Forest Tuning ===")
print("Best params:", grid_rf.best_params_)
print("Best CV Accuracy:", round(grid_rf.best_score_, 4))

### PCA + K-Means Clustering

#### Apply PCA to reduce service ratings to 2 dimensions

In [ ]:
from sklearn.decomposition import PCA

# PCA on the 14 service rating columns
pca = PCA(n_components=2)
service_pca_train = pca.fit_transform(X_train_scaled[service_cols])

print("Variance explained by PC1:", round(pca.explained_variance_ratio_[0], 4))
print("Variance explained by PC2:", round(pca.explained_variance_ratio_[1], 4))
print("Total variance captured:", round(sum(pca.explained_variance_ratio_), 4))

#### Elbow Method to find the best number of clusters

In [ ]:
from sklearn.cluster import KMeans

inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(service_pca_train)
    inertias.append(km.inertia_)

plt.plot(range(2, 11), inertias, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

#### Fit K-Means with chosen k and visualize

In [ ]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(service_pca_train)

# Scatter plot colored by cluster
plt.figure(figsize=(10, 6))
scatter = plt.scatter(service_pca_train[:, 0], service_pca_train[:, 1], 
                      c=cluster_labels, cmap='viridis', alpha=0.3, s=5)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means Clusters (PCA)')
plt.show()

#### Analyze what each cluster looks like

In [ ]:
train['Cluster'] = cluster_labels

# Profile each cluster
cluster_profile = train.groupby('Cluster').agg({
    'satisfaction': 'mean',
    'Avg Service Rating': 'mean',
    'Class': 'mean',
    'Age': 'mean',
    'Flight Distance': 'mean',
    'Total Delay': 'mean'
}).round(3)

cluster_profile.columns = ['Satisfaction Rate', 'Avg Service', 'Avg Class', 'Avg Age', 'Avg Distance', 'Avg Delay']
print(cluster_profile)

# Count passengers per cluster
print("\nPassengers per cluster:")
print(train['Cluster'].value_counts().sort_index())

### Summary table of all models

In [ ]:
y_pred_best_lr = grid_lr.best_estimator_.predict(X_test_scaled)
y_pred_best_dt = grid_dt.best_estimator_.predict(X_test_scaled)
y_pred_best_rf = grid_rf.best_estimator_.predict(X_test_scaled)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_best_lr), 4),
        round(accuracy_score(y_test, y_pred_best_dt), 4),
        round(accuracy_score(y_test, y_pred_best_rf), 4)
    ],
    'ROC-AUC': [
        round(roc_auc_score(y_test, grid_lr.best_estimator_.predict_proba(X_test_scaled)[:,1]), 4),
        round(roc_auc_score(y_test, grid_dt.best_estimator_.predict_proba(X_test_scaled)[:,1]), 4),
        round(roc_auc_score(y_test, grid_rf.best_estimator_.predict_proba(X_test_scaled)[:,1]), 4)
    ]
})

print(results.to_string(index=False))

#### Feature importance from Random Forest

In [ ]:
# Top 10 most important features
importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': grid_rf.best_estimator_.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importances)
plt.title('Top 10 Most Important Features (Random Forest)')
plt.tight_layout()
plt.show()

#### ROC curves for all 3 models on one plot

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))

for name, model in [('Logistic Regression', grid_lr.best_estimator_), 
                     ('Decision Tree', grid_dt.best_estimator_), 
                     ('Random Forest', grid_rf.best_estimator_)]:
    probs = model.predict_proba(X_test_scaled)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')

plt.plot([0,1], [0,1], 'k--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend()
plt.show()

#### Confusion Matrix (Random Forest)

In [2]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_best_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Dissatisfied', 'Satisfied'])

plt.figure(figsize=(8, 6))
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Random Forest')
plt.show()

print(f"True Negatives: {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives: {cm[1][1]}")

ModuleNotFoundError: No module named 'sklearn'

In [3]:
comparison = pd.DataFrame({
    'Actual': y_test.values[:20],
    'Logistic Regression': y_pred_best_lr[:20],
    'Decision Tree': y_pred_best_dt[:20],
    'Random Forest': y_pred_best_rf[:20]
})

comparison['Actual Label'] = comparison['Actual'].map({1: 'Satisfied', 0: 'Dissatisfied'})
comparison['RF Prediction'] = comparison['Random Forest'].map({1: 'Satisfied', 0: 'Dissatisfied'})
comparison['Correct?'] = comparison['Actual'] == comparison['Random Forest']

print(comparison.to_string(index=False))

NameError: name 'y_test' is not defined

##### Testing with example data

In [4]:
# Create a fake passenger
import numpy as np

new_passenger = pd.DataFrame({
    'Gender': [1],                              # Male
    'Customer Type': [0],                       # Loyal
    'Age': [35],
    'Type of Travel': [0],                      # Business
    'Class': [0],                               # Business class
    'Flight Distance': [1500],
    'Inflight wifi service': [4],
    'Departure/Arrival time convenient': [4],
    'Ease of Online booking': [3],
    'Gate location': [3],
    'Food and drink': [5],
    'Online boarding': [5],
    'Seat comfort': [5],
    'Inflight entertainment': [4],
    'On-board service': [4],
    'Leg room service': [4],
    'Baggage handling': [4],
    'Checkin service': [4],
    'Inflight service': [4],
    'Cleanliness': [5],
    'Departure Delay in Minutes': [0],
    'Arrival Delay in Minutes': [0]
})

new_passenger['Total Delay'] = new_passenger['Departure Delay in Minutes'] + new_passenger['Arrival Delay in Minutes']
new_passenger['Avg Service Rating'] = new_passenger[service_cols].mean(axis=1)

# Scale using the same scaler
new_scaled = pd.DataFrame(scaler.transform(new_passenger), columns=X_train.columns)

# Predict with all 3 models
print("Logistic Regression says:", "Satisfied" if grid_lr.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")
print("Decision Tree says:", "Satisfied" if grid_dt.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")
print("Random Forest says:", "Satisfied" if grid_rf.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")

# Show probability
prob = grid_rf.best_estimator_.predict_proba(new_scaled)[0]
print(f"\nRandom Forest confidence: {prob[1]*100:.1f}% chance satisfied")

NameError: name 'scaler' is not defined

##### Example 2

In [ ]:
new_passenger = pd.DataFrame({
    'Gender': [0],
    'Customer Type': [1],                       # Disloyal
    'Age': [25],
    'Type of Travel': [1],                      # Personal travel
    'Class': [1],                               # Economy
    'Flight Distance': [500],
    'Inflight wifi service': [1],
    'Departure/Arrival time convenient': [1],
    'Ease of Online booking': [1],
    'Gate location': [2],
    'Food and drink': [1],
    'Online boarding': [1],
    'Seat comfort': [2],
    'Inflight entertainment': [1],
    'On-board service': [2],
    'Leg room service': [1],
    'Baggage handling': [2],
    'Checkin service': [2],
    'Inflight service': [1],
    'Cleanliness': [2],
    'Departure Delay in Minutes': [120],
    'Arrival Delay in Minutes': [130]
})

new_passenger['Total Delay'] = new_passenger['Departure Delay in Minutes'] + new_passenger['Arrival Delay in Minutes']
new_passenger['Avg Service Rating'] = new_passenger[service_cols].mean(axis=1)

new_scaled = pd.DataFrame(scaler.transform(new_passenger), columns=X_train.columns)

print("Logistic Regression:", "Satisfied" if grid_lr.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")
print("Decision Tree:", "Satisfied" if grid_dt.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")
print("Random Forest:", "Satisfied" if grid_rf.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")

prob = grid_rf.best_estimator_.predict_proba(new_scaled)[0]
print(f"Random Forest confidence: {prob[0]*100:.1f}% chance dissatisfied")

##### Example 3

In [ ]:
new_passenger = pd.DataFrame({
    'Gender': [0],                              # Female
    'Customer Type': [0],                       # Loyal
    'Age': [45],
    'Type of Travel': [0],                      # Business travel
    'Class': [0],                               # Business class
    'Flight Distance': [2500],
    'Inflight wifi service': [5],
    'Departure/Arrival time convenient': [5],
    'Ease of Online booking': [5],
    'Gate location': [3],
    'Food and drink': [5],
    'Online boarding': [5],
    'Seat comfort': [5],
    'Inflight entertainment': [5],
    'On-board service': [5],
    'Leg room service': [5],
    'Baggage handling': [5],
    'Checkin service': [5],
    'Inflight service': [5],
    'Cleanliness': [5],
    'Departure Delay in Minutes': [0],
    'Arrival Delay in Minutes': [0]
})

new_passenger['Total Delay'] = new_passenger['Departure Delay in Minutes'] + new_passenger['Arrival Delay in Minutes']
new_passenger['Avg Service Rating'] = new_passenger[service_cols].mean(axis=1)

new_scaled = pd.DataFrame(scaler.transform(new_passenger), columns=X_train.columns)

print("Logistic Regression:", "Satisfied" if grid_lr.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")
print("Decision Tree:", "Satisfied" if grid_dt.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")
print("Random Forest:", "Satisfied" if grid_rf.best_estimator_.predict(new_scaled)[0] == 1 else "Dissatisfied")

prob = grid_rf.best_estimator_.predict_proba(new_scaled)[0]
print(f"\nRandom Forest confidence: {prob[1]*100:.1f}% chance satisfied")